# Two tower recommender: walkthrough of the results

This notebook reads the files written by `run_experiment.py`. Run that first:

```bash
python run_experiment.py --download
```

All metrics come from simulated viewers and measure how well a model recovers the simulator's rules, not quality for real viewers.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ART = ROOT / "artifacts"
assert (ART / "hypothesis_tests.json").exists(), "Run run_experiment.py first"

## 1. Data schema and relationships

Two files with the same columns (movies also carry budget and revenue). One row per title. There is no viewer, event or rating table, and `show_id` is only unique within a file.

In [ ]:
pd.read_csv(ART / "eda_schema.csv")

## 2. Basic statistics and the findings behind the design

In [ ]:
summary = json.loads((ART / "eda_summary.json").read_text())
{k: v for k, v in summary.items() if k != "missing_pct_by_type"}

In [ ]:
display(Image(ART / "eda_missingness.png"))
display(Image(ART / "eda_english_share_by_popularity.png"))
display(Image(ART / "eda_languages.png"))

## 3. Research design

The hypotheses and decision rules are in `HYPOTHESES.md`, committed before the final run.

1. **H1**: description text adds signal beyond genre and metadata.
2. **H2**: a popularity ranking shows far fewer non-English titles than viewers watch; the two tower shows about as many as viewers watch.

## 4. Every model run

In [ ]:
runs = pd.read_csv(ART / "metrics_all_runs.csv")
cols = ["model", "pop_exponent", "seed", "recall@10", "ndcg@10", "coverage@10", "non_en@10", "tv@10"]
runs[cols].round(4)

## 5. Hypothesis tests

In [ ]:
tests = json.loads((ART / "hypothesis_tests.json").read_text())
print("H1:", tests["H1"]["verdict"])
print("H2a:", tests["H2"]["h2a_verdict"], "| H2b:", tests["H2"]["h2b_verdict"])
for name in ("h1_text_ablation.png", "h2_non_english_gap.png"):
    if (ART / name).exists():
        display(Image(ART / name))

In [ ]:
text = (ART / "results_summary.md").read_text()
display(Markdown(text[text.index("## H1"):]))

## 6. Sample recommendations for three hand-seeded viewers

In [ ]:
display(Markdown((ART / "sample_recommendations.md").read_text()))

## 7. Interpretation

Strengths, weaknesses and the production path are discussed in `RESULTS.md` and the README.